<a href="https://colab.research.google.com/github/harshkumar8a/fine-tuning-using-mlflow/blob/main/Fine_Tuning_with_Mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer
import transformers
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import torch


In [5]:
print(transformers.__version__)

5.0.0


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
# Load IMDb sentiment dataset (25k train, 25k test, labels: 0=neg, 1=pos)
dataset = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [10]:
# Load BERT tokenizer
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",   # pad shorter sequences
        truncation=True,         # cut sequences > 512 tokens
        max_length=128           # BERT max is 512; 128 is faster for demos
    )

# Apply tokenization to all splits
tokenized = dataset.map(tokenize_fn, batched=True)

# Remove the raw text column — Trainer only needs token IDs
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")  # Trainer expects "labels"
tokenized.set_format("torch")  # Return PyTorch tensors

print(tokenized["train"][0].keys())
# dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [12]:
# Load pre-trained BERT with a fresh 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # negative / positive
)
model.to(device)
print(model.device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


In [13]:
# Metric function called at the end of each eval epoch
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,            # small LR prevents catastrophic forgetting
    weight_decay=0.01,             # L2 regularization

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)


In [15]:
# Use only 2000 samples for speed in this demo
small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
small_eval  = tokenized["test"].shuffle(seed=42).select(range(500))

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.403419,0.832000,0.830701
2,No log,0.423883,0.832000,0.830450
3,No log,0.397847,0.860000,0.859827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=375, training_loss=0.33953076171875, metrics={'train_runtime': 187.8765, 'train_samples_per_second': 31.936, 'train_steps_per_second': 1.996, 'total_flos': 394666583040000.0, 'train_loss': 0.33953076171875, 'epoch': 3.0})

## Using MLflow

In [18]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 26.3 MB/s eta 0:00:00


In [ ]:
pip install urllib3


In [ ]:
import mlflow
import mlflow.pytorch
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import urllib3

In [ ]:
CONFIG = {
    "model_name":   "bert-base-uncased",
    "num_labels":   2,
    "max_length":   128,
    "batch_size":   16,
    "epochs":       3,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "train_size":   2000,
    "eval_size":    500,
}

In [ ]:
dataset   = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length",
                     truncation=True, max_length=CONFIG["max_length"])

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
train_ds = tokenized["train"].shuffle(seed=42).select(range(CONFIG["train_size"]))
eval_ds  = tokenized["test"].shuffle(seed=42).select(range(CONFIG["eval_size"]))

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")     # local MLflow server
mlflow.set_experiment("bert-sentiment-classification") # logical grouping

KeyboardInterrupt: 

In [ ]:
with mlflow.start_run(run_name="bert-base-lr2e5-ep3") as run:

    # Log every hyperparameter — you'll thank yourself when comparing runs
    mlflow.log_params(CONFIG)

    # Tag run with context metadata
    mlflow.set_tags({
        "dataset":     "imdb",
        "task":        "sentiment-classification",
        "framework":   "huggingface-transformers",
        "developer":   "Harsh Kumar",
    })

    # Build model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_labels"]
    )

    training_args = TrainingArguments(
        output_dir       = f"./results/{run.info.run_id}",  # unique per run!
        num_train_epochs = CONFIG["epochs"],
        per_device_train_batch_size = CONFIG["batch_size"],
        per_device_eval_batch_size  = CONFIG["batch_size"] * 2,
        learning_rate    = CONFIG["learning_rate"],
        weight_decay     = CONFIG["weight_decay"],
        evaluation_strategy = "epoch",
        save_strategy       = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model  = "f1",
        report_to = "none",   # disable default reporting; we log manually
    )

    from transformers import TrainerCallback

    class MLflowMetricsCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics:
                epoch = int(state.epoch)
                mlflow.log_metric("eval_accuracy", metrics.get("eval_accuracy", 0), step=epoch)
                mlflow.log_metric("eval_f1",       metrics.get("eval_f1", 0),       step=epoch)
                mlflow.log_metric("eval_loss",     metrics.get("eval_loss", 0),     step=epoch)

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                mlflow.log_metric("train_loss", logs["loss"], step=int(state.global_step))

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = eval_ds,
        compute_metrics = compute_metrics,
        callbacks       = [MLflowMetricsCallback()],
    )

    trainer.train()

    # ── 6. Final evaluation ───────────────────────────────────────────────────
    final_metrics = trainer.evaluate()
    mlflow.log_metrics({
        "final_accuracy": final_metrics["eval_accuracy"],
        "final_f1":       final_metrics["eval_f1"],
    })

    # ── 7. Save model as MLflow artifact ─────────────────────────────────────
    # Option A: save as HuggingFace pipeline (easy to load later)
    from transformers import pipeline
    clf_pipeline = pipeline(
        "text-classification",
        model=trainer.model,
        tokenizer=tokenizer
    )
    mlflow.transformers.log_model(
        transformers_model=clf_pipeline,
        artifact_path="model",              # folder name inside the run
        task="text-classification",
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Final F1: {final_metrics['eval_f1']:.4f}")
    print(f"Final Accuracy: {final_metrics['eval_accuracy']:.4f}")

In [ ]:


# ── 3. Metrics ────────────────────────────────────────────────────────────────


# ── 4. MLflow experiment setup ────────────────────────────────────────────────


# ── 5. Training inside an MLflow run ─────────────────────────────────────────




    # Custom callback to log epoch metrics to MLflow


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
mlflow ui
